
# Eye tracking + text2text + TabR (Optuna, GPU)

This notebook mirrors the structure of `experiments_eye_text2text_index_match.ipynb`, but replaces tree models with a **TabR-style retrieval-augmented tabular deep model** trained on **GPU** and tuned with **Optuna**.

Main differences from the original notebook:
- uses a **TabR-like PyTorch model** instead of XGBoost / CatBoost
- tunes **model parameters and training parameters jointly**
- runs **grouped CV** on the training split
- saves each experiment to a **separate folder**
- keeps the same **eye-tracking + text2text index-matching** logic

Notes:
- the implementation is intentionally self-contained inside the notebook
- retrieval is performed on GPU with PyTorch tensors
- the code assumes all features are numerical after preprocessing / concatenation


In [1]:

from __future__ import annotations

import os
import gc
import json
import math
import random
from copy import deepcopy
from pathlib import Path
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
from umap import UMAP

import optuna
from optuna.samplers import TPESampler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import GroupKFold, StratifiedKFold

from experiment_code.read_data import get_data_for_split
from experiment_code.run_experiments import _build_groups_mm_for_mm


# Data root: contains datasets_splitted_screen_match_mismatch*
ROOT = Path(r"C:\Users\LEGION\data\СB")
os.chdir(ROOT)
print("cwd:", Path.cwd())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("cuda:", torch.cuda.get_device_name(0))


cwd: C:\Users\LEGION\data\СB
device: cuda
cuda: NVIDIA GeForce RTX 4080 Laptop GPU


In [2]:

# Main settings
TARGETS = ["match_mismatch", "match_mismatch_general"]
SPLITS = ["binary", "multiclass"]

CV = 4
N_TRIALS_TABR = 60
RANDOM_STATE = 42

USE_UMAP_FOR = {"freq_bands", "stat", "corr", "cov_freq"}
UMAP_N_COMPONENTS = 100
CLEAN_TRAIN_COLUMNS = False

MAX_EPOCHS_DEFAULT = 100
EARLY_STOPPING_PATIENCE_DEFAULT = 20
N_WORKERS = 0

# Output location under project folder
PROJECT_DIR = Path(r"C:\Users\LEGION\Projects\CB_exepriment\dataset_v2")
OUT_DIR = PROJECT_DIR / "optuna_results_eye_text2text_tabr"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS_DIR = OUT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRIALS_DIR = OUT_DIR / "optuna_trials"
TRIALS_DIR.mkdir(parents=True, exist_ok=True)

PREDS_DIR = OUT_DIR / "predictions"
PREDS_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_DIR = OUT_DIR / "summaries"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

TEXT2TEXT_DIR = Path(r"C:\Users\LEGION\Projects\CB_exepriment\text2text_features")
TEXT2TEXT_FILES = {
    "corr": TEXT2TEXT_DIR / "corr.parquet",
    "cov_freq": TEXT2TEXT_DIR / "cov_freq.parquet",
    "envelope": TEXT2TEXT_DIR / "envelope.parquet",
    "freq_bands": TEXT2TEXT_DIR / "freq_bands.parquet",
    "PID": TEXT2TEXT_DIR / "PID.parquet",
    "stat": TEXT2TEXT_DIR / "stat.parquet",
}


In [3]:

def seed_everything(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def _prefix_cols(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return df.add_prefix(prefix)


def _load_text2text_parquet(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    if "pid_rn" not in df.columns:
        raise KeyError(f"'pid_rn' column not found in {path}")
    df = df.set_index("pid_rn")
    df.index = df.index.astype(str)
    return df


def _apply_umap_train_test(
    text_train: pd.DataFrame,
    text_test: pd.DataFrame,
    *,
    text_set_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    tr = text_train.apply(pd.to_numeric, errors="coerce")
    te = text_test.apply(pd.to_numeric, errors="coerce")

    train_means = tr.mean(numeric_only=True)
    tr = tr.fillna(train_means).fillna(0.0)
    te = te.fillna(train_means).fillna(0.0)

    n_components = int(min(UMAP_N_COMPONENTS, max(2, tr.shape[1])))
    reducer = UMAP(n_components=n_components, random_state=RANDOM_STATE)

    print(
        f"[{text_set_name}] UMAP compression: "
        f"train/test features {tr.shape[1]} -> {n_components}"
    )

    z_train = reducer.fit_transform(tr)
    z_test = reducer.transform(te)

    cols = [f"{text_set_name}__umap_{i:03d}" for i in range(n_components)]
    tr_umap = pd.DataFrame(z_train, index=text_train.index, columns=cols)
    te_umap = pd.DataFrame(z_test, index=text_test.index, columns=cols)
    return tr_umap, te_umap


def _prepare_joined_X(
    eye_train: pd.DataFrame,
    eye_test: pd.DataFrame,
    text_all: pd.DataFrame,
    text_set_name: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    text_train = text_all.reindex(eye_train.index)
    text_test = text_all.reindex(eye_test.index)

    missing_train = int(text_train.isna().all(axis=1).sum())
    missing_test = int(text_test.isna().all(axis=1).sum())
    print(f"[{text_set_name}] rows missing after index match -> train: {missing_train}, test: {missing_test}")

    if text_set_name in USE_UMAP_FOR:
        text_train_model, text_test_model = _apply_umap_train_test(
            text_train,
            text_test,
            text_set_name=text_set_name,
        )
    else:
        text_train_model = _prefix_cols(text_train, f"{text_set_name}__")
        text_test_model = _prefix_cols(text_test, f"{text_set_name}__")

    X_train = pd.concat([
        _prefix_cols(eye_train, "eye__"),
        text_train_model,
    ], axis=1)
    X_test = pd.concat([
        _prefix_cols(eye_test, "eye__"),
        text_test_model,
    ], axis=1)

    if CLEAN_TRAIN_COLUMNS:
        keep_cols = X_train.columns[X_train.notna().all(axis=0)]
        X_train = X_train[keep_cols]
        X_test = X_test.reindex(columns=keep_cols)

    # numeric-only, fit preprocessing on train only
    X_train = X_train.apply(pd.to_numeric, errors="coerce")
    X_test = X_test.apply(pd.to_numeric, errors="coerce")

    means = X_train.mean(axis=0)
    stds = X_train.std(axis=0).replace(0.0, 1.0)

    X_train = X_train.fillna(means).fillna(0.0)
    X_test = X_test.fillna(means).fillna(0.0)

    X_train = (X_train - means) / stds
    X_test = (X_test - means) / stds

    X_train = X_train.replace([np.inf, -np.inf], 0.0)
    X_test = X_test.replace([np.inf, -np.inf], 0.0)

    return X_train.astype(np.float32), X_test.astype(np.float32)


def _build_no_neutral_mask(stim_df: pd.DataFrame, target: str, split: str) -> pd.Series:
    mask = pd.Series(True, index=stim_df.index)

    if "valence" in stim_df.columns:
        valence = pd.to_numeric(stim_df["valence"], errors="coerce")
        mask &= valence != 2

    if split == "binary":
        if target == "match_mismatch_general" and "exp_general_multi" in stim_df.columns:
            exp_general_multi = pd.to_numeric(stim_df["exp_general_multi"], errors="coerce")
            mask &= exp_general_multi != 2
        elif "IAT_results2" in stim_df.columns:
            iat = stim_df["IAT_results2"].astype(str).str.strip().str.lower()
            mask &= iat != "neutral"

    return mask.fillna(False)


def _load_eye_for_task(target: str, split: str, no_neutral: bool = False):
    df, tgt, _ = get_data_for_split(X_name="screen", target=target, split=split)

    eye_train = df["all_features"]["X_train"].copy()
    eye_test = df["all_features"]["X_test"].copy()
    stim_train = df["stimuli_features"]["X_train"].copy()
    stim_test = df["stimuli_features"]["X_test"].copy()

    y_train = tgt["cb"]["y_train"].copy()
    y_test = tgt["cb"]["y_test"].copy()

    if no_neutral:
        mask_train = _build_no_neutral_mask(stim_train, target=target, split=split)
        mask_test = _build_no_neutral_mask(stim_test, target=target, split=split)

        eye_train = eye_train.loc[mask_train]
        eye_test = eye_test.loc[mask_test]
        y_train = y_train.loc[mask_train]
        y_test = y_test.loc[mask_test]
        stim_train = stim_train.loc[mask_train]

    groups = np.array([str(i).split("_")[0] for i in eye_train.index])

    if target in ("match_mismatch", "match_mismatch_general"):
        groups_mm = _build_groups_mm_for_mm(X_train_index=eye_train.index, stim_train_df=stim_train)
    else:
        groups_mm = None

    problem = "binary" if split == "binary" else "multiclass"
    return eye_train, eye_test, y_train, y_test, groups, groups_mm, problem


In [4]:

class NumpyTabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        if y.ndim != 1:
            y = y.reshape(-1)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class TabRModel(nn.Module):
    """
    Simplified TabR-style model for numerical tabular data.

    Architecture:
    1. encoder -> latent representation z
    2. key projection for retrieval
    3. retrieve top-k neighbours from candidate set (train fold)
    4. aggregate label-aware context
    5. predictor head

    Designed to stay close in spirit to the official TabR structure while being
    self-contained and easy to tune in a notebook.
    """
    def __init__(
        self,
        input_dim: int,
        n_classes: int,
        d_main: int = 256,
        d_block: int = 512,
        encoder_n_blocks: int = 2,
        predictor_n_blocks: int = 2,
        context_dropout: float = 0.1,
        dropout0: float = 0.1,
        dropout1: float = 0.1,
        normalization: str = "LayerNorm",
        activation: str = "ReLU",
        k_neighbors: int = 32,
        label_smoothing: float = 0.0,
    ):
        super().__init__()
        self.n_classes = n_classes
        self.k_neighbors = k_neighbors
        self.label_smoothing = label_smoothing

        Norm = getattr(nn, normalization)
        Act = getattr(nn, activation)

        self.input = nn.Linear(input_dim, d_main)

        def make_block():
            return nn.Sequential(
                Norm(d_main),
                nn.Linear(d_main, d_block),
                Act(),
                nn.Dropout(dropout0),
                nn.Linear(d_block, d_main),
                nn.Dropout(dropout1),
            )

        self.encoder_blocks = nn.ModuleList([make_block() for _ in range(encoder_n_blocks)])
        self.norm_before_retrieval = Norm(d_main)
        self.key_proj = nn.Linear(d_main, d_main)
        self.context_transform = nn.Sequential(
            nn.Linear(d_main, d_block),
            Act(),
            nn.Dropout(dropout0),
            nn.Linear(d_block, d_main, bias=False),
        )
        self.context_dropout = nn.Dropout(context_dropout)

        self.label_encoder = nn.Embedding(n_classes, d_main)

        self.predictor_blocks = nn.ModuleList([make_block() for _ in range(predictor_n_blocks)])
        self.head = nn.Sequential(
            Norm(d_main),
            Act(),
            nn.Linear(d_main, n_classes),
        )

    def encode(self, x: torch.Tensor):
        z = self.input(x)
        for block in self.encoder_blocks:
            z = z + block(z)
        k = self.key_proj(self.norm_before_retrieval(z))
        return z, k

    def retrieve_context(
        self,
        query_k: torch.Tensor,
        cand_z: torch.Tensor,
        cand_k: torch.Tensor,
        cand_y: torch.Tensor,
    ) -> torch.Tensor:
        # negative squared distance
        # shapes: query [B, D], cand [N, D]
        # gpu-friendly exact retrieval for moderate-size folds
        q2 = (query_k ** 2).sum(dim=1, keepdim=True)          # [B, 1]
        c2 = (cand_k ** 2).sum(dim=1).unsqueeze(0)            # [1, N]
        sim = -(q2 - 2.0 * query_k @ cand_k.T + c2)          # [B, N]

        k = min(self.k_neighbors, cand_k.shape[0])
        top_sim, top_idx = torch.topk(sim, k=k, dim=1)

        probs = F.softmax(top_sim, dim=1)
        probs = self.context_dropout(probs)

        neigh_z = cand_z[top_idx]                            # [B, k, D]
        neigh_y = cand_y[top_idx]                            # [B, k]
        neigh_y_emb = self.label_encoder(neigh_y)            # [B, k, D]

        values = neigh_y_emb + self.context_transform(query_k[:, None, :] - neigh_z)
        context = (probs.unsqueeze(1) @ values).squeeze(1)   # [B, D]
        return context

    def forward(
        self,
        x: torch.Tensor,
        candidate_x: torch.Tensor,
        candidate_y: torch.Tensor,
    ) -> torch.Tensor:
        z, k = self.encode(x)
        cand_z, cand_k = self.encode(candidate_x)
        context = self.retrieve_context(k, cand_z, cand_k, candidate_y)
        h = z + context
        for block in self.predictor_blocks:
            h = h + block(h)
        return self.head(h)


In [5]:

def make_loaders(X_train, y_train, batch_size, shuffle=True):
    ds = NumpyTabularDataset(X_train, y_train)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=N_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )


def predict_proba_tabr(
    model: nn.Module,
    X_eval: np.ndarray,
    X_candidates: np.ndarray,
    y_candidates: np.ndarray,
    batch_size: int = 1024,
) -> np.ndarray:
    model.eval()

    X_cand_t = torch.tensor(X_candidates, dtype=torch.float32, device=DEVICE)
    y_cand_t = torch.tensor(y_candidates, dtype=torch.long, device=DEVICE)

    all_logits = []
    with torch.no_grad():
        for start in range(0, len(X_eval), batch_size):
            xb = torch.tensor(X_eval[start:start+batch_size], dtype=torch.float32, device=DEVICE)
            logits = model(xb, X_cand_t, y_cand_t)
            all_logits.append(logits.detach().cpu())
    logits = torch.cat(all_logits, dim=0)
    probs = torch.softmax(logits, dim=1).numpy()
    return probs


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }


def train_one_fold(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    params: dict,
    seed: int = RANDOM_STATE,
):
    seed_everything(seed)

    model = TabRModel(
        input_dim=X_tr.shape[1],
        n_classes=int(len(np.unique(y_tr))),
        d_main=params["d_main"],
        d_block=params["d_block"],
        encoder_n_blocks=params["encoder_n_blocks"],
        predictor_n_blocks=params["predictor_n_blocks"],
        context_dropout=params["context_dropout"],
        dropout0=params["dropout0"],
        dropout1=params["dropout1"],
        normalization=params["normalization"],
        activation=params["activation"],
        k_neighbors=params["k_neighbors"],
        label_smoothing=params["label_smoothing"],
    ).to(DEVICE)

    optimizer_name = params["optimizer"]
    lr = params["lr"]
    weight_decay = params["weight_decay"]

    if optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.RAdam(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=max(4, params["patience"] // 3),
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=params["label_smoothing"])

    loader = make_loaders(X_tr, y_tr, params["batch_size"], shuffle=True)

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=DEVICE)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long, device=DEVICE)

    best_state = None
    best_score = -np.inf
    best_epoch = -1
    no_improve = 0
    history = []

    for epoch in range(params["max_epochs"]):
        model.train()
        epoch_losses = []

        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb, X_tr_t, y_tr_t)
            loss = criterion(logits, yb)
            loss.backward()

            if params["grad_clip"] is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), params["grad_clip"])

            optimizer.step()
            epoch_losses.append(loss.item())

        val_probs = predict_proba_tabr(
            model=model,
            X_eval=X_val,
            X_candidates=X_tr,
            y_candidates=y_tr,
            batch_size=params["eval_batch_size"],
        )
        val_pred = val_probs.argmax(axis=1)
        val_metrics = compute_metrics(y_val, val_pred)
        val_score = val_metrics["macro_f1"]
        scheduler.step(val_score)

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(epoch_losses)),
            **{f"val_{k}": v for k, v in val_metrics.items()}
        })

        if val_score > best_score:
            best_score = val_score
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= params["patience"]:
            break

    if best_state is None:
        raise RuntimeError("Training finished without a best checkpoint.")

    model.load_state_dict(best_state)
    return model, best_score, best_epoch, history


In [6]:

def suggest_tabr_params(trial: optuna.Trial, n_features: int, n_classes: int) -> dict:
    d_main = trial.suggest_categorical("d_main", [128, 192, 256, 384, 512])
    d_mult = trial.suggest_categorical("d_mult", [1.5, 2.0, 2.5, 3.0, 4.0])

    params = {
        # model
        "d_main": d_main,
        "d_block": int(d_main * d_mult),
        "encoder_n_blocks": trial.suggest_int("encoder_n_blocks", 1, 4),
        "predictor_n_blocks": trial.suggest_int("predictor_n_blocks", 1, 4),
        "context_dropout": trial.suggest_float("context_dropout", 0.0, 0.4),
        "dropout0": trial.suggest_float("dropout0", 0.0, 0.4),
        "dropout1": trial.suggest_float("dropout1", 0.0, 0.4),
        "normalization": trial.suggest_categorical("normalization", ["LayerNorm"]),
        "activation": trial.suggest_categorical("activation", ["ReLU", "GELU"]),
        "k_neighbors": trial.suggest_categorical("k_neighbors", [8, 16, 24, 32, 48, 64]),

        # training
        "optimizer": trial.suggest_categorical("optimizer", ["adamw", "adam", "radam"]),
        "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 5e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256, 512, 1024]),
        "eval_batch_size": trial.suggest_categorical("eval_batch_size", [256, 512, 1024, 2048]),
        "max_epochs": trial.suggest_int("max_epochs", 80, MAX_EPOCHS_DEFAULT),
        "patience": trial.suggest_int("patience", 12, EARLY_STOPPING_PATIENCE_DEFAULT),
        "grad_clip": trial.suggest_categorical("grad_clip", [None, 1.0, 2.0, 5.0]),
        "label_smoothing": trial.suggest_float("label_smoothing", 0.0, 0.1),
    }
    return params


def run_cv_optuna(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    *,
    study_name: str,
    n_trials: int = N_TRIALS_TABR,
    cv: int = CV,
    seed: int = RANDOM_STATE,
):
    X_np = X.to_numpy(dtype=np.float32)
    y_np = np.asarray(y).astype(int)

    def objective(trial: optuna.Trial) -> float:
        params = suggest_tabr_params(trial, n_features=X_np.shape[1], n_classes=len(np.unique(y_np)))
        splitter = GroupKFold(n_splits=cv)

        fold_scores = []
        for fold_idx, (tr_idx, va_idx) in enumerate(splitter.split(X_np, y_np, groups=groups)):
            X_tr, X_va = X_np[tr_idx], X_np[va_idx]
            y_tr, y_va = y_np[tr_idx], y_np[va_idx]

            model, best_score, best_epoch, history = train_one_fold(
                X_tr=X_tr,
                y_tr=y_tr,
                X_val=X_va,
                y_val=y_va,
                params=params,
                seed=seed + fold_idx,
            )
            fold_scores.append(best_score)

            trial.report(np.mean(fold_scores), step=fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

            del model
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        return float(np.mean(fold_scores))

    sampler = TPESampler(seed=seed, multivariate=True)
    study = optuna.create_study(
        direction="maximize",
        study_name=study_name,
        sampler=sampler,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    trials_df = study.trials_dataframe()
    return study, trials_df


In [7]:

def fit_best_and_test_tabr(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    best_params: dict,
    *,
    seed: int = RANDOM_STATE,
):
    X_train_np = X_train.to_numpy(dtype=np.float32)
    X_test_np = X_test.to_numpy(dtype=np.float32)
    y_train_np = np.asarray(y_train).astype(int)
    y_test_np = np.asarray(y_test).astype(int)

    # hold out a small validation part from train for early stopping
    val_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, va_idx = next(val_splitter.split(X_train_np, y_train_np))

    X_tr, X_va = X_train_np[tr_idx], X_train_np[va_idx]
    y_tr, y_va = y_train_np[tr_idx], y_train_np[va_idx]

    model, best_score, best_epoch, history = train_one_fold(
        X_tr=X_tr,
        y_tr=y_tr,
        X_val=X_va,
        y_val=y_va,
        params=best_params,
        seed=seed,
    )

    test_probs = predict_proba_tabr(
        model=model,
        X_eval=X_test_np,
        X_candidates=X_train_np,
        y_candidates=y_train_np,
        batch_size=best_params["eval_batch_size"],
    )
    test_pred = test_probs.argmax(axis=1)
    test_metrics = compute_metrics(y_test_np, test_pred)

    return {
        "model": model,
        "best_score_internal_val": float(best_score),
        "best_epoch": int(best_epoch),
        "history": history,
        "test_metrics": test_metrics,
        "test_pred": test_pred,
        "test_probs": test_probs,
    }


In [8]:

def run_text2text_set_tabr(
    text_set_name: str,
    text_all: pd.DataFrame,
    *,
    n_trials: int = N_TRIALS_TABR,
    do_test_fit: bool = True,
):
    if "pid_rn" in text_all.columns:
        text_all = text_all.set_index("pid_rn")
    text_all = text_all.copy()
    text_all.index = text_all.index.astype(str)

    if "key" in text_all.columns:
        text_all = text_all.drop(columns=["key"])

    text_features = int(text_all.shape[1])
    print(f"[{text_set_name}] text2text feature count after cleanup: {text_features}")

    rows = []

    for target in TARGETS:
        for split in SPLITS:
            for no_neutral in [False, True]:
                problem = "binary" if split == "binary" else "multiclass"
                neutral_suffix = "__no_neutral" if no_neutral else ""
                exp_name = (
                    f"tabr__X_name=screen+{text_set_name}"
                    f"__target={target}"
                    f"__problem={problem}"
                    f"__feat=all_features+{text_set_name}"
                    f"__gpu_optuna"
                    f"{neutral_suffix}"
                )
                exp_dir = OUT_DIR / exp_name
                exp_dir.mkdir(parents=True, exist_ok=True)

                report_path = exp_dir / "report.json"
                if report_path.exists():
                    print("\n" + "=" * 80)
                    print(f"Skipping existing experiment: {exp_name}")
                    with open(report_path, "r", encoding="utf-8") as f:
                        report = json.load(f)
                    rows.append({
                        "text_set": text_set_name,
                        "target": target,
                        "split": split,
                        "no_neutral": no_neutral,
                        "tabr_test_macro_f1": report["test_metrics"]["macro_f1"],
                        "tabr_test_accuracy": report["test_metrics"]["accuracy"],
                        "file": str(report_path),
                    })
                    continue

                eye_train, eye_test, y_train, y_test, groups, groups_mm, problem = _load_eye_for_task(
                    target,
                    split,
                    no_neutral=no_neutral,
                )
                X_train, X_test = _prepare_joined_X(eye_train, eye_test, text_all, text_set_name)

                print("\n" + "=" * 80)
                print(f"Running: {exp_name}")
                print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

                study, trials_df = run_cv_optuna(
                    X=X_train,
                    y=y_train,
                    groups=groups,
                    study_name=exp_name,
                    n_trials=n_trials,
                    cv=CV,
                    seed=RANDOM_STATE,
                )

                best_params = study.best_trial.params
                if "d_block" not in best_params:
                    best_params["d_block"] = int(best_params["d_main"] * best_params["d_mult"])
                with open(exp_dir / "best_params.json", "w", encoding="utf-8") as f:
                    json.dump(best_params, f, indent=2, ensure_ascii=False)

                trials_df.to_csv(exp_dir / "optuna_trials.csv", index=False)

                final_out = fit_best_and_test_tabr(
                    X_train=X_train,
                    y_train=y_train,
                    X_test=X_test,
                    y_test=y_test,
                    best_params=best_params,
                    seed=RANDOM_STATE,
                )

                pred_df = pd.DataFrame({
                    "index": X_test.index.astype(str),
                    "y_true": np.asarray(y_test).astype(int),
                    "y_pred": final_out["test_pred"].astype(int),
                })
                for i in range(final_out["test_probs"].shape[1]):
                    pred_df[f"proba_{i}"] = final_out["test_probs"][:, i]
                pred_df.to_csv(exp_dir / "test_predictions.csv", index=False)

                torch.save(final_out["model"].state_dict(), exp_dir / "tabr_model.pt")

                report = {
                    "experiment_name": exp_name,
                    "text_set": text_set_name,
                    "target": target,
                    "split": split,
                    "problem": problem,
                    "n_train": int(X_train.shape[0]),
                    "n_test": int(X_test.shape[0]),
                    "n_features": int(X_train.shape[1]),
                    "device": str(DEVICE),
                    "best_trial_number": int(study.best_trial.number),
                    "best_cv_macro_f1": float(study.best_value),
                    "best_params": best_params,
                    "best_epoch_internal_val": int(final_out["best_epoch"]),
                    "test_metrics": final_out["test_metrics"],
                    "report_file": str(report_path),
                }

                with open(report_path, "w", encoding="utf-8") as f:
                    json.dump(report, f, indent=2, ensure_ascii=False)

                rows.append({
                    "text_set": text_set_name,
                    "target": target,
                    "split": split,
                    "no_neutral": no_neutral,
                    "tabr_test_macro_f1": float(final_out["test_metrics"]["macro_f1"]),
                    "tabr_test_accuracy": float(final_out["test_metrics"]["accuracy"]),
                    "file": str(report_path),
                })

                del final_out
                gc.collect()
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()

    summary = pd.DataFrame(rows)
    display(summary)
    return summary


In [9]:

print("text2text dir:", TEXT2TEXT_DIR)
for k, p in TEXT2TEXT_FILES.items():
    print(f"{k:10s} -> {p} | exists={p.exists()}")
    if p.exists():
        print(pd.read_parquet(p).shape)


text2text dir: C:\Users\LEGION\Projects\CB_exepriment\text2text_features
corr       -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\corr.parquet | exists=True
(5592, 1832)
cov_freq   -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\cov_freq.parquet | exists=True
(5592, 36297)
envelope   -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\envelope.parquet | exists=True
(5592, 429)
freq_bands -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\freq_bands.parquet | exists=True
(5592, 18363)
PID        -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\PID.parquet | exists=True
(5592, 612)
stat       -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\stat.parquet | exists=True
(5592, 8237)


In [10]:

# Set: corr
text_corr = pd.read_parquet(TEXT2TEXT_FILES["corr"])
print("corr shape:", text_corr.shape)
summary_corr = run_text2text_set_tabr("corr", text_corr)
summary_corr


corr shape: (5592, 1832)
[corr] text2text feature count after cleanup: 1830

Skipping existing experiment: tabr__X_name=screen+corr__target=match_mismatch__problem=binary__feat=all_features+corr__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+corr__target=match_mismatch__problem=binary__feat=all_features+corr__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+corr__target=match_mismatch__problem=multiclass__feat=all_features+corr__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+corr__target=match_mismatch__problem=multiclass__feat=all_features+corr__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+corr__target=match_mismatch_general__problem=binary__feat=all_features+corr__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+corr__target=match_mismatch_general__problem=binary__feat=all_features+corr__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+corr__target=match_mismatch

,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,corr,match_mismatch,binary,False,0.508172,0.550976,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,corr,match_mismatch,binary,True,0.425768,0.538690,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,corr,match_mismatch,multiclass,False,0.567961,0.555315,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,corr,match_mismatch,multiclass,True,0.544316,0.544643,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,corr,match_mismatch_general,binary,False,0.506735,0.577007,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,corr,match_mismatch_general,binary,True,0.477679,0.535714,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,corr,match_mismatch_general,multiclass,False,0.649632,0.650759,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,corr,match_mismatch_general,multiclass,True,0.582120,0.589286,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,corr,match_mismatch,binary,False,0.508172,0.550976,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,corr,match_mismatch,binary,True,0.425768,0.538690,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,corr,match_mismatch,multiclass,False,0.567961,0.555315,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,corr,match_mismatch,multiclass,True,0.544316,0.544643,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,corr,match_mismatch_general,binary,False,0.506735,0.577007,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,corr,match_mismatch_general,binary,True,0.477679,0.535714,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,corr,match_mismatch_general,multiclass,False,0.649632,0.650759,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,corr,match_mismatch_general,multiclass,True,0.582120,0.589286,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [11]:

# Set: cov_freq
text_cov_freq = pd.read_parquet(TEXT2TEXT_FILES["cov_freq"])
print("cov_freq shape:", text_cov_freq.shape)
summary_cov_freq = run_text2text_set_tabr("cov_freq", text_cov_freq)
summary_cov_freq


cov_freq shape: (5592, 36297)
[cov_freq] text2text feature count after cleanup: 36295

Skipping existing experiment: tabr__X_name=screen+cov_freq__target=match_mismatch__problem=binary__feat=all_features+cov_freq__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+cov_freq__target=match_mismatch__problem=binary__feat=all_features+cov_freq__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+cov_freq__target=match_mismatch__problem=multiclass__feat=all_features+cov_freq__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+cov_freq__target=match_mismatch__problem=multiclass__feat=all_features+cov_freq__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+cov_freq__target=match_mismatch_general__problem=binary__feat=all_features+cov_freq__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+cov_freq__target=match_mismatch_general__problem=binary__feat=all_features+cov_freq__gpu_optuna__no_neutral

Skipping existing e

,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,cov_freq,match_mismatch,binary,False,0.492957,0.548807,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,cov_freq,match_mismatch,binary,True,0.468639,0.577381,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,cov_freq,match_mismatch,multiclass,False,0.552524,0.535792,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,cov_freq,match_mismatch,multiclass,True,0.498864,0.500000,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,cov_freq,match_mismatch_general,binary,False,0.557298,0.622560,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,cov_freq,match_mismatch_general,binary,True,0.559892,0.654762,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,cov_freq,match_mismatch_general,multiclass,False,0.621884,0.622560,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,cov_freq,match_mismatch_general,multiclass,True,0.520323,0.526786,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,cov_freq,match_mismatch,binary,False,0.492957,0.548807,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,cov_freq,match_mismatch,binary,True,0.468639,0.577381,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,cov_freq,match_mismatch,multiclass,False,0.552524,0.535792,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,cov_freq,match_mismatch,multiclass,True,0.498864,0.500000,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,cov_freq,match_mismatch_general,binary,False,0.557298,0.622560,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,cov_freq,match_mismatch_general,binary,True,0.559892,0.654762,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,cov_freq,match_mismatch_general,multiclass,False,0.621884,0.622560,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,cov_freq,match_mismatch_general,multiclass,True,0.520323,0.526786,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [12]:

# Set: envelope
text_envelope = pd.read_parquet(TEXT2TEXT_FILES["envelope"])
print("envelope shape:", text_envelope.shape)
summary_envelope = run_text2text_set_tabr("envelope", text_envelope)
summary_envelope


envelope shape: (5592, 429)
[envelope] text2text feature count after cleanup: 427

Skipping existing experiment: tabr__X_name=screen+envelope__target=match_mismatch__problem=binary__feat=all_features+envelope__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+envelope__target=match_mismatch__problem=binary__feat=all_features+envelope__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+envelope__target=match_mismatch__problem=multiclass__feat=all_features+envelope__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+envelope__target=match_mismatch__problem=multiclass__feat=all_features+envelope__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+envelope__target=match_mismatch_general__problem=binary__feat=all_features+envelope__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+envelope__target=match_mismatch_general__problem=binary__feat=all_features+envelope__gpu_optuna__no_neutral

Skipping existing exper

,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,envelope,match_mismatch,binary,False,0.499941,0.581345,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,envelope,match_mismatch,binary,True,0.403539,0.520833,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,envelope,match_mismatch,multiclass,False,0.535088,0.524946,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,envelope,match_mismatch,multiclass,True,0.502972,0.502976,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,envelope,match_mismatch_general,binary,False,0.552176,0.609544,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,envelope,match_mismatch_general,binary,True,0.538310,0.601190,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,envelope,match_mismatch_general,multiclass,False,0.587256,0.583514,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,envelope,match_mismatch_general,multiclass,True,0.603547,0.607143,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,envelope,match_mismatch,binary,False,0.499941,0.581345,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,envelope,match_mismatch,binary,True,0.403539,0.520833,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,envelope,match_mismatch,multiclass,False,0.535088,0.524946,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,envelope,match_mismatch,multiclass,True,0.502972,0.502976,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,envelope,match_mismatch_general,binary,False,0.552176,0.609544,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,envelope,match_mismatch_general,binary,True,0.538310,0.601190,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,envelope,match_mismatch_general,multiclass,False,0.587256,0.583514,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,envelope,match_mismatch_general,multiclass,True,0.603547,0.607143,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [13]:

# Set: freq_bands
text_freq_bands = pd.read_parquet(TEXT2TEXT_FILES["freq_bands"])
print("freq_bands shape:", text_freq_bands.shape)
summary_freq_bands = run_text2text_set_tabr("freq_bands", text_freq_bands)
summary_freq_bands


freq_bands shape: (5592, 18363)
[freq_bands] text2text feature count after cleanup: 18361

Skipping existing experiment: tabr__X_name=screen+freq_bands__target=match_mismatch__problem=binary__feat=all_features+freq_bands__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+freq_bands__target=match_mismatch__problem=binary__feat=all_features+freq_bands__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+freq_bands__target=match_mismatch__problem=multiclass__feat=all_features+freq_bands__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+freq_bands__target=match_mismatch__problem=multiclass__feat=all_features+freq_bands__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+freq_bands__target=match_mismatch_general__problem=binary__feat=all_features+freq_bands__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+freq_bands__target=match_mismatch_general__problem=binary__feat=all_features+freq_bands__gpu_optuna__no_

,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,freq_bands,match_mismatch,binary,False,0.444435,0.503254,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,freq_bands,match_mismatch,binary,True,0.485601,0.592262,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,freq_bands,match_mismatch,multiclass,False,0.555486,0.537961,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,freq_bands,match_mismatch,multiclass,True,0.494224,0.497024,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,freq_bands,match_mismatch_general,binary,False,0.527205,0.581345,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,freq_bands,match_mismatch_general,binary,True,0.547881,0.642857,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,freq_bands,match_mismatch_general,multiclass,False,0.693793,0.687636,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,freq_bands,match_mismatch_general,multiclass,True,0.663218,0.666667,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,freq_bands,match_mismatch,binary,False,0.444435,0.503254,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,freq_bands,match_mismatch,binary,True,0.485601,0.592262,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,freq_bands,match_mismatch,multiclass,False,0.555486,0.537961,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,freq_bands,match_mismatch,multiclass,True,0.494224,0.497024,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,freq_bands,match_mismatch_general,binary,False,0.527205,0.581345,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,freq_bands,match_mismatch_general,binary,True,0.547881,0.642857,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,freq_bands,match_mismatch_general,multiclass,False,0.693793,0.687636,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,freq_bands,match_mismatch_general,multiclass,True,0.663218,0.666667,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [14]:

# Set: PID
text_PID = pd.read_parquet(TEXT2TEXT_FILES["PID"])
print("PID shape:", text_PID.shape)
summary_PID = run_text2text_set_tabr("PID", text_PID)
summary_PID


PID shape: (5592, 612)
[PID] text2text feature count after cleanup: 610

Skipping existing experiment: tabr__X_name=screen+PID__target=match_mismatch__problem=binary__feat=all_features+PID__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+PID__target=match_mismatch__problem=binary__feat=all_features+PID__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+PID__target=match_mismatch__problem=multiclass__feat=all_features+PID__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+PID__target=match_mismatch__problem=multiclass__feat=all_features+PID__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+PID__target=match_mismatch_general__problem=binary__feat=all_features+PID__gpu_optuna

Skipping existing experiment: tabr__X_name=screen+PID__target=match_mismatch_general__problem=binary__feat=all_features+PID__gpu_optuna__no_neutral

Skipping existing experiment: tabr__X_name=screen+PID__target=match_mismatch_general__problem

c:\Users\LEGION\miniconda3\envs\table_methods\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-04-14 10:57:38,697] A new study created in memory with name: tabr__X_name=screen+PID__target=match_mismatch_general__problem=multiclass__feat=all_features+PID__gpu_optuna__no_neutral



Running: tabr__X_name=screen+PID__target=match_mismatch_general__problem=multiclass__feat=all_features+PID__gpu_optuna__no_neutral
X_train: (3016, 1080), X_test: (336, 1080)


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-04-14 10:57:49,992] Trial 0 finished with value: 0.5442200352056208 and parameters: {'d_main': 192, 'd_mult': 2.5, 'encoder_n_blocks': 1, 'predictor_n_blocks': 4, 'context_dropout': 0.3329770563201687, 'dropout0': 0.08493564427131046, 'dropout1': 0.07272998688284026, 'normalization': 'LayerNorm', 'activation': 'GELU', 'k_neighbors': 32, 'optimizer': 'radam', 'lr': 0.00021839352923182988, 'weight_decay': 7.982478599323911e-05, 'batch_size': 512, 'eval_batch_size': 1024, 'max_epochs': 86, 'patience': 12, 'grad_clip': None, 'label_smoothing': 0.0034388521115218396}. Best is trial 0 with value: 0.5442200352056208.
[I 2026-04-14 10:58:09,801] Trial 1 finished with value: 0.6000166409453915 and parameters: {'d_main': 128, 'd_mult': 2.5, 'encoder_n_blocks': 4, 'predictor_n_blocks': 3, 'context_dropout': 0.36874969400924673, 'dropout0': 0.0353970008207678, 'dropout1': 0.07839314496765809, 'normalization': 'LayerNorm', 'activation': 'GELU', 'k_neighbors': 24, 'optimizer': 'adam', 'lr': 

,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,PID,match_mismatch,binary,False,0.509131,0.570499,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,PID,match_mismatch,binary,True,0.412698,0.559524,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,PID,match_mismatch,multiclass,False,0.591670,0.585683,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,PID,match_mismatch,multiclass,True,0.526278,0.526786,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,PID,match_mismatch_general,binary,False,0.512506,0.568330,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,PID,match_mismatch_general,binary,True,0.479882,0.556548,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,PID,match_mismatch_general,multiclass,False,0.581319,0.581345,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,PID,match_mismatch_general,multiclass,True,0.554460,0.556548,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,PID,match_mismatch,binary,False,0.509131,0.570499,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,PID,match_mismatch,binary,True,0.412698,0.559524,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,PID,match_mismatch,multiclass,False,0.591670,0.585683,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,PID,match_mismatch,multiclass,True,0.526278,0.526786,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,PID,match_mismatch_general,binary,False,0.512506,0.568330,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,PID,match_mismatch_general,binary,True,0.479882,0.556548,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,PID,match_mismatch_general,multiclass,False,0.581319,0.581345,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,PID,match_mismatch_general,multiclass,True,0.554460,0.556548,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [15]:

# Set: stat
text_stat = pd.read_parquet(TEXT2TEXT_FILES["stat"])
print("stat shape:", text_stat.shape)
summary_stat = run_text2text_set_tabr("stat", text_stat)
summary_stat


stat shape: (5592, 8237)
[stat] text2text feature count after cleanup: 8235
[stat] rows missing after index match -> train: 1193, test: 131
[stat] UMAP compression: train/test features 8235 -> 100


c:\Users\LEGION\miniconda3\envs\table_methods\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\LEGION\miniconda3\envs\table_methods\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-04-14 11:16:08,693] A new study created in memory with name: tabr__X_name=screen+stat__target=match_mismatch__problem=binary__feat=all_features+stat__gpu_optuna



Running: tabr__X_name=screen+stat__target=match_mismatch__problem=binary__feat=all_features+stat__gpu_optuna
X_train: (4146, 570), X_test: (461, 570)


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-04-14 11:16:17,174] Trial 0 finished with value: 0.4964346509868236 and parameters: {'d_main': 192, 'd_mult': 2.5, 'encoder_n_blocks': 1, 'predictor_n_blocks': 4, 'context_dropout': 0.3329770563201687, 'dropout0': 0.08493564427131046, 'dropout1': 0.07272998688284026, 'normalization': 'LayerNorm', 'activation': 'GELU', 'k_neighbors': 32, 'optimizer': 'radam', 'lr': 0.00021839352923182988, 'weight_decay': 7.982478599323911e-05, 'batch_size': 512, 'eval_batch_size': 1024, 'max_epochs': 86, 'patience': 12, 'grad_clip': None, 'label_smoothing': 0.0034388521115218396}. Best is trial 0 with value: 0.4964346509868236.
[I 2026-04-14 11:16:30,331] Trial 1 finished with value: 0.5023656254183245 and parameters: {'d_main': 128, 'd_mult': 2.5, 'encoder_n_blocks': 4, 'predictor_n_blocks': 3, 'context_dropout': 0.36874969400924673, 'dropout0': 0.0353970008207678, 'dropout1': 0.07839314496765809, 'normalization': 'LayerNorm', 'activation': 'GELU', 'k_neighbors': 24, 'optimizer': 'adam', 'lr': 

...


Running: tabr__X_name=screen+stat__target=match_mismatch_general__problem=multiclass__feat=all_features+stat__gpu_optuna__no_neutral
X_train: (3016, 570), X_test: (336, 570)


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-04-14 13:22:43,582] Trial 0 finished with value: 0.5376339458329044 and parameters: {'d_main': 192, 'd_mult': 2.5, 'encoder_n_blocks': 1, 'predictor_n_blocks': 4, 'context_dropout': 0.3329770563201687, 'dropout0': 0.08493564427131046, 'dropout1': 0.07272998688284026, 'normalization': 'LayerNorm', 'activation': 'GELU', 'k_neighbors': 32, 'optimizer': 'radam', 'lr': 0.00021839352923182988, 'weight_decay': 7.982478599323911e-05, 'batch_size': 512, 'eval_batch_size': 1024, 'max_epochs': 86, 'patience': 12, 'grad_clip': None, 'label_smoothing': 0.0034388521115218396}. Best is trial 0 with value: 0.5376339458329044.
[I 2026-04-14 13:23:07,563] Trial 1 finished with value: 0.5863621482839685 and parameters: {'d_main': 128, 'd_mult': 2.5, 'encoder_n_blocks': 4, 'predictor_n_blocks': 3, 'context_dropout': 0.36874969400924673, 'dropout0': 0.0353970008207678, 'dropout1': 0.07839314496765809, 'normalization': 'LayerNorm', 'activation': 'GELU', 'k_neighbors': 24, 'optimizer': 'adam', 'lr': 

,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,stat,match_mismatch,binary,False,0.427984,0.529284,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,stat,match_mismatch,binary,True,0.435704,0.476190,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,stat,match_mismatch,multiclass,False,0.580387,0.568330,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,stat,match_mismatch,multiclass,True,0.452381,0.452381,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,stat,match_mismatch_general,binary,False,0.575839,0.622560,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,stat,match_mismatch_general,binary,True,0.510204,0.595238,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,stat,match_mismatch_general,multiclass,False,0.662470,0.659436,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,stat,match_mismatch_general,multiclass,True,0.642384,0.645833,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,stat,match_mismatch,binary,False,0.427984,0.529284,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,stat,match_mismatch,binary,True,0.435704,0.476190,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,stat,match_mismatch,multiclass,False,0.580387,0.568330,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,stat,match_mismatch,multiclass,True,0.452381,0.452381,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,stat,match_mismatch_general,binary,False,0.575839,0.622560,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,stat,match_mismatch_general,binary,True,0.510204,0.595238,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,stat,match_mismatch_general,multiclass,False,0.662470,0.659436,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,stat,match_mismatch_general,multiclass,True,0.642384,0.645833,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [16]:

all_summaries = [
    summary_corr,
    summary_cov_freq,
    summary_envelope,
    summary_freq_bands,
    summary_PID,
    summary_stat,
]
all_results = pd.concat(all_summaries, ignore_index=True)
all_results


,text_set,target,split,no_neutral,tabr_test_macro_f1,tabr_test_accuracy,file
0,corr,match_mismatch,binary,False,0.508172,0.550976,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,corr,match_mismatch,binary,True,0.425768,0.538690,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,corr,match_mismatch,multiclass,False,0.567961,0.555315,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,corr,match_mismatch,multiclass,True,0.544316,0.544643,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,corr,match_mismatch_general,binary,False,0.506735,0.577007,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,corr,match_mismatch_general,binary,True,0.477679,0.535714,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,corr,match_mismatch_general,multiclass,False,0.649632,0.650759,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,corr,match_mismatch_general,multiclass,True,0.582120,0.589286,C:\Users\LEGION\Projects\CB_exepriment\dataset...
8,cov_freq,match_mismatch,binary,False,0.492957,0.548807,C:\Users\LEGION\Projects\CB_exepriment\dataset...
9,cov_freq,match_mismatch,binary,True,0.468639,0.577381,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [17]:

summary_path = OUT_DIR / "summary_eye_text2text_tabr.csv"
all_results.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)

all_results.groupby(["text_set", "target", "split"], as_index=False)[["tabr_test_macro_f1", "tabr_test_accuracy"]].mean()


Saved summary: C:\Users\LEGION\Projects\CB_exepriment\dataset_v2\optuna_results_eye_text2text_tabr\summary_eye_text2text_tabr.csv


,text_set,target,split,tabr_test_macro_f1,tabr_test_accuracy
0,PID,match_mismatch,binary,0.460915,0.565011
1,PID,match_mismatch,multiclass,0.558974,0.556235
2,PID,match_mismatch_general,binary,0.496194,0.562439
3,PID,match_mismatch_general,multiclass,0.567889,0.568946
4,corr,match_mismatch,binary,0.466970,0.544833
5,corr,match_mismatch,multiclass,0.556139,0.549979
6,corr,match_mismatch_general,binary,0.492207,0.556360
7,corr,match_mismatch_general,multiclass,0.615876,0.620022
8,cov_freq,match_mismatch,binary,0.480798,0.563094
9,cov_freq,match_mismatch,multiclass,0.525694,0.517896
